# Edge detection comparison: classical vs HED vs PiDiNet (paper-pretrained and our own)

Runs **one image** through 11 methods and plots the outputs side by side:

| Group | Methods | Runs on |
|---|---|---|
| Classical | Sobel, Roberts, Canny | CPU (OpenCV / scikit-image, 1-70 ms) |
| Deep (2015) | HED (original Caffe weights, loaded into the repo's PyTorch HED) | **GPU** |
| Paper-pretrained | Baseline, PiDiNet-Tiny, PiDiNet-Small, PiDiNet-Full | **GPU** |
| **Ours (retrained)** | Baseline, PiDiNet-Full (28,800 real BSDS pairs, 3 epochs) | **GPU** |

For the paper's own README images, a 12th panel shows the authors' published output.

**Kernel:** pick the Python environment where you installed the requirements (see the repo README), top-right of the notebook in VS Code. Open the notebook from the repo's `notebook/` folder: it finds the sibling `src/` folder by itself, so no paths need editing.

## 1. Setup

In [ ]:
import os, sys, time, subprocess
from types import SimpleNamespace

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from skimage.filters import roberts

def find_repo():
    """The repo's src/ folder (main.py, models/, trained_models/, custom_images/ ...), found relative to
    this notebook so it works from a fresh clone. Jupyter/VS Code start the kernel in the notebook's folder."""
    here = os.getcwd()
    for cand in (os.path.join(here, "..", "src"), os.path.join(here, "src"), here):
        if os.path.isfile(os.path.join(cand, "main.py")) and os.path.isdir(os.path.join(cand, "models")):
            return os.path.abspath(cand)
    raise FileNotFoundError("Could not find the src/ folder (with main.py and models/). "
                            "Open this notebook from the repo's notebook/ folder, or set REPO by hand.")

REPO = find_repo()
print("repo src folder:", REPO)
sys.path.insert(0, REPO)
os.chdir(REPO)

import models                                        # PiDiNet model zoo from the repo
from models.convert_pidinet import convert_pidinet   # PDC -> plain conv folding
from models.hed_fps import HED                       # PyTorch HED architecture from the repo

OUT_DIR = os.path.join(REPO, "results", "notebook_compare")
os.makedirs(OUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
print("torch", torch.__version__, "| opencv", cv2.__version__)

## 2. Classical detectors (Sobel, Roberts, Canny)
No training involved: fixed hand-designed kernels, run on CPU. Sobel/Roberts get a light Gaussian blur;
Canny's thresholds come from each image's own gradient distribution.

In [ ]:
def to_u8(x):
    """Rescale any float map to 0..255 uint8 (edges bright on dark)."""
    x = x.astype(np.float32)
    hi = np.percentile(x, 99.0)                        # clip the brightest 1% so faint edges stay visible
    return (255 * np.clip(x / (hi + 1e-8), 0, 1)).astype(np.uint8)

def run_sobel(gray):
    g = cv2.GaussianBlur(gray, (3, 3), 0)
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    return to_u8(np.hypot(gx, gy))

def run_roberts(gray):
    g = cv2.GaussianBlur(gray, (3, 3), 0) / 255.0
    return to_u8(roberts(g))

def run_canny(gray, hi_pct=95, lo_ratio=0.4):
    """Canny with thresholds taken from the image's own gradient distribution, so low-contrast
    photos are not handicapped by a fixed threshold. Output is binary 0/255."""
    g = cv2.GaussianBlur(gray, (5, 5), 0)
    mag = np.abs(cv2.Sobel(g, cv2.CV_32F, 1, 0, 3)) + np.abs(cv2.Sobel(g, cv2.CV_32F, 0, 1, 3))
    hi = np.percentile(mag, hi_pct)
    return cv2.Canny(g, lo_ratio * hi, hi)

## 3. HED (Holistically-Nested Edge Detection, 2015) on the GPU
The original pretrained Caffe weights (~59 MB, downloaded once into `src/hed_weights/`).
OpenCV (CPU-only here) reads the Caffe file, and we copy each layer's weights into the repo's PyTorch
`HED` network (`models/hed_fps.py`) so inference runs on the GPU. The OpenCV network is kept as a CPU reference.

In [ ]:
HED_DIR   = os.path.join(REPO, "hed_weights"); os.makedirs(HED_DIR, exist_ok=True)
HED_MODEL = os.path.join(HED_DIR, "hed_pretrained_bsds.caffemodel")
HED_PROTO = os.path.join(HED_DIR, "deploy.prototxt")

def fetch(url, dst, min_bytes):
    if os.path.exists(dst) and os.path.getsize(dst) >= min_bytes:
        return
    # --ssl-no-revoke: this Windows box fails the certificate-revocation lookup (see slide 13)
    subprocess.run(["curl.exe", "-sSL", "--ssl-no-revoke", "-o", dst, url], check=True)
    assert os.path.getsize(dst) >= min_bytes, f"download incomplete: {dst}"

fetch("https://vcl.ucsd.edu/hed/hed_pretrained_bsds.caffemodel", HED_MODEL, 58_000_000)
fetch("https://raw.githubusercontent.com/s9xie/hed/master/examples/hed/deploy.prototxt", HED_PROTO, 5_000)

class CropLayer:
    """Center-crop layer used by HED's deconv outputs (OpenCV has no built-in version)."""
    def __init__(self, params, blobs):
        self.ystart = self.yend = self.xstart = self.xend = 0
    def getMemoryShapes(self, inputs):
        inp, tgt = inputs[0], inputs[1]
        self.ystart = (inp[2] - tgt[2]) // 2
        self.xstart = (inp[3] - tgt[3]) // 2
        self.yend, self.xend = self.ystart + tgt[2], self.xstart + tgt[3]
        return [[inp[0], inp[1], tgt[2], tgt[3]]]
    def forward(self, inputs):
        return [inputs[0][:, :, self.ystart:self.yend, self.xstart:self.xend]]

try:
    cv2.dnn_registerLayer("Crop", CropLayer)
except cv2.error:
    pass                                  # already registered (re-running this cell)
hed_net = cv2.dnn.readNetFromCaffe(HED_PROTO, HED_MODEL)       # CPU; weight source + reference

HED_MEAN_BGR = (104.00698793, 116.66876762, 122.67891434)

# PyTorch layer name -> Caffe layer name in deploy.prototxt
CAFFE_NAME = {n: n for n in ["conv1_1", "conv1_2", "conv2_1", "conv2_2", "conv3_1", "conv3_2", "conv3_3",
                             "conv4_1", "conv4_2", "conv4_3", "conv5_1", "conv5_2", "conv5_3"]}
CAFFE_NAME.update({"score_dsn1": "score-dsn1-up", "score_dsn2": "score-dsn2", "score_dsn3": "score-dsn3",
                   "score_dsn4": "score-dsn4", "score_dsn5": "score-dsn5", "score_final": "upscore-fuse"})

def build_hed_gpu():
    m = HED()
    for py_name, caffe_name in CAFFE_NAME.items():
        w, b = hed_net.getLayer(hed_net.getLayerId(caffe_name)).blobs      # [weight, bias]
        layer = getattr(m, py_name)
        layer.weight.data.copy_(torch.from_numpy(w.copy()))
        layer.bias.data.copy_(torch.from_numpy(b.reshape(-1).copy()))
    m = m.to(device).eval()
    for a in ["weight_deconv2", "weight_deconv3", "weight_deconv4", "weight_deconv5"]:   # fixed bilinear filters
        setattr(m, a, getattr(m, a).to(device))
    return m

hed_gpu = build_hed_gpu()

@torch.no_grad()
def run_hed(bgr):
    """HED on the GPU: BGR 0..255 minus the Caffe mean, sigmoid of the fused map."""
    x = (bgr.astype(np.float32) - np.array(HED_MEAN_BGR, dtype=np.float32)).transpose(2, 0, 1)[None]
    y = torch.sigmoid(hed_gpu(torch.from_numpy(x).to(device)))[0, 0]
    return (255 * y.cpu().numpy()).clip(0, 255).astype(np.uint8)

def run_hed_cpu(bgr):
    """Reference: the same weights run by OpenCV's DNN on the CPU."""
    H, W = bgr.shape[:2]
    blob = cv2.dnn.blobFromImage(bgr, 1.0, (W, H), HED_MEAN_BGR, swapRB=False, crop=False)
    hed_net.setInput(blob)
    return (255 * cv2.resize(hed_net.forward()[0, 0], (W, H))).clip(0, 255).astype(np.uint8)

### HED sanity check: GPU port vs OpenCV on the same weights
They are two implementations of the same network (OpenCV uses a plain centre crop, the PyTorch port HED's
aligned crop), so expect a high but not perfect correlation.

In [ ]:
bgr = cv2.imread(os.path.join(REPO, "custom_images", "lcd4inch.jpg"))
a, b = run_hed(bgr), run_hed_cpu(bgr)
print("correlation GPU-vs-OpenCV: %.3f" % np.corrcoef(a.ravel().astype(float), b.ravel().astype(float))[0, 1])

## 4. PiDiNet: paper-pretrained and our own checkpoints (GPU)
* `converted=True` folds every pixel-difference conv into one plain conv (`convert_pidinet`, slide 4), so the
  network runs as an ordinary CNN. The `baseline` config is already vanilla conv, so it needs no conversion.
* Checkpoints were saved from `DataParallel`, so keys start with `module.`. We strip that prefix.
* The model returns 5 maps (4 stage outputs + fused); the **last one** is the final edge probability map.

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def load_pidinet(arch, config, ckpt, converted):
    args = SimpleNamespace(config=config, sa=True, dil=True)          # CSAM + CDCM on, as in the paper
    model = getattr(models, arch + ("_converted" if converted else ""))(args)
    state = torch.load(os.path.join(REPO, ckpt), map_location="cpu", weights_only=False)["state_dict"]
    state = {k.replace("module.", "", 1): v for k, v in state.items()}
    if converted:
        state = convert_pidinet(state, config)
    model.load_state_dict(state)
    return model.to(device).eval()

@torch.no_grad()
def run_pidinet(model, rgb):
    x = ((rgb.astype(np.float32) / 255.0 - MEAN) / STD).transpose(2, 0, 1)[None]
    y = model(torch.from_numpy(x).to(device))[-1]                      # fused edge map, sigmoid already applied
    return (255 * y[0, 0].cpu().numpy()).clip(0, 255).astype(np.uint8)

# (label, architecture, config, checkpoint, converted)
PIDINET_MODELS = [
    ("Baseline (paper)",      "pidinet",       "baseline", "trained_models/table5_baseline.pth",      False),
    ("PiDiNet-Tiny (paper)",  "pidinet_tiny",  "carv4",    "trained_models/table5_pidinet-tiny.pth",  True),
    ("PiDiNet-Small (paper)", "pidinet_small", "carv4",    "trained_models/table5_pidinet-small.pth", True),
    ("PiDiNet-Full (paper)",  "pidinet",       "carv4",    "trained_models/table5_pidinet.pth",       True),
    ("OURS Baseline",         "pidinet",       "baseline", "results/train_baseline_real/save_models/checkpoint_002.pth", False),
    ("OURS PiDiNet-Full",     "pidinet",       "carv4",    "results/train_pidinet_real/save_models/checkpoint_002.pth",  True),
]
LOADED = {label: load_pidinet(a, c, k, v) for label, a, c, k, v in PIDINET_MODELS}
print("loaded:", list(LOADED))

## 5. Run every method on an image and plot
Each panel title shows inference time (average of 3 runs after a warm-up). Sobel/Roberts/Canny run on CPU;
HED and the PiDiNet models run on the GPU. If `paper_ref` is given, it is added as a 12th panel.
Individual output maps are also saved to `results/notebook_compare/<image>/`.

In [ ]:
def timed(fn, *a, n=3):
    fn(*a)                                                    # warm-up
    t = time.perf_counter()
    for _ in range(n):
        out = fn(*a)
    return out, (time.perf_counter() - t) / n * 1000.0        # ms per image (outputs are copied to CPU, so GPU work is synced)

def compare_image(image_path, show=True, paper_ref=None, tag=None):
    bgr  = cv2.imread(image_path); assert bgr is not None, f"cannot read {image_path}"
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    outs, ms = {}, {}
    for name, fn, arg in [("Sobel", run_sobel, gray), ("Roberts", run_roberts, gray),
                          ("Canny", run_canny, gray), ("HED (2015)", run_hed, bgr)]:
        outs[name], ms[name] = timed(fn, arg)
    for label, model in LOADED.items():
        outs[label], ms[label] = timed(run_pidinet, model, rgb)

    stem = tag or os.path.splitext(os.path.basename(image_path))[0]
    os.makedirs(os.path.join(OUT_DIR, stem), exist_ok=True)
    for name, img in outs.items():
        safe = name.replace(" ", "_").replace("(", "").replace(")", "")
        cv2.imwrite(os.path.join(OUT_DIR, stem, safe + ".png"), img)

    panels = [("Original", rgb, None)] + [(n, o, ms[n]) for n, o in outs.items()]
    if paper_ref is not None:
        panels.append(("Paper's published output\n(authors' README figure)", paper_ref, None))
    aspect = bgr.shape[1] / bgr.shape[0]                       # width / height of the photo
    cols = 6 if aspect < 1.0 else 4                            # portrait photos: 6 across so the figure stays wide
    rows = -(-len(panels) // cols)
    row_h = float(np.clip(3.6 / aspect + 0.7, 2.4, 6.0))       # row height follows the photo's shape (no wide gaps)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.6, rows * row_h))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (name, img, t) in zip(axes.ravel(), panels):
        ax.imshow(img, cmap=None if img.ndim == 3 else "gray", vmin=0, vmax=255)
        ours = name.startswith("OURS")
        ax.set_title(name if t is None else f"{name}\n{t:.0f} ms", fontsize=11,
                     color="#0E7C74" if ours else "black", fontweight="bold" if ours else "normal")
    fig.suptitle(f"{stem}   |   Sobel/Roberts/Canny on CPU; HED + PiDiNet models on {device}", fontsize=12)
    fig.tight_layout()
    out_png = os.path.join(OUT_DIR, f"compare_{stem}.png")
    fig.savefig(out_png, dpi=140, bbox_inches="tight")
    if show: plt.show()
    else: plt.close(fig)
    print("saved:", out_png)
    return outs, ms

## 6. Run on one image  (change the path to use your own)

In [ ]:
IMAGE = os.path.join(REPO, "custom_images", "lcd4inch.jpg")
outs, ms = compare_image(IMAGE)

## 7. All four test photos (GPU)

In [ ]:
for name in ["lcd4inch", "20260919103052", "labubu", "naruto"]:
    compare_image(os.path.join(REPO, "custom_images", f"{name}.jpg"))

## 8. The paper's own images
The PiDiNet README shows two photos with the authors' published edge maps. We download that figure, crop the
input photos and the authors' outputs, run all 11 methods on the same photos, and show the authors' output as
a 12th panel so it can be compared with our run of the same checkpoint (`PiDiNet-Full (paper)`).

**Resolution matters.** The README figure only contains ~520x390 thumbnails, but the authors most likely ran
the full-resolution phone photos. At native size our maps look thicker and softer than the published ones; a plain
bicubic x3 upscale (no new detail, just a larger input) gives maps that closely match the published output.
We run both sizes.

In [ ]:
from PIL import Image
PAPER_DIR = os.path.join(REPO, "paper_demo"); os.makedirs(PAPER_DIR, exist_ok=True)
ref_path = os.path.join(PAPER_DIR, "readme_figure.jpg")
if not os.path.exists(ref_path) or os.path.getsize(ref_path) < 10_000:
    subprocess.run(["curl.exe", "-sSL", "--ssl-no-revoke", "-o", ref_path,
                    "https://user-images.githubusercontent.com/18327074/129970337-bb467a8c-825e-47ee-872c-533f0a5da37a.jpg"],
                   check=True)
fig_img = Image.open(ref_path).convert("RGB")                  # 1196 x 801: photos on the left, authors' outputs on the right
crops = {   # name: (input box, authors' output box)
    "paper_office_desk": ((0, 0, 520, 390),   (675, 0, 1196, 390)),
    "paper_lake":        ((0, 410, 520, 800), (675, 410, 1196, 800)),
}
paper_ref = {}
for name, (in_box, out_box) in crops.items():
    fig_img.crop(in_box).save(os.path.join(PAPER_DIR, name + ".jpg"), quality=95)
    paper_ref[name] = np.array(fig_img.crop(out_box).convert("L"))
for name in crops:
    for scale in (1, 3):                                       # native thumbnail, then x3 bicubic upscale
        path = os.path.join(PAPER_DIR, name + ".jpg")
        if scale > 1:
            im = Image.open(path)
            path = os.path.join(PAPER_DIR, f"{name}_x{scale}.jpg")
            im.resize((im.width * scale, im.height * scale), Image.BICUBIC).save(path, quality=95)
        compare_image(path, paper_ref=paper_ref[name], tag=f"{name}_x{scale}")